# 영화 추천: 검색

**Two-Tower Model** (1. Query Model(user)/ 2. Candidate Model(movie)) ==> 두 모델의 출력할 함께 곱한다.

대략적인 ANN 인덱스를 구축

*목표: 코드 이해/ 내부 작동 이해*

In [64]:
import os

import pprint
import tempfile

from typing import Dict, Text

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

In [65]:
os.environ["TF_USE_LEGACY_KERAS"] = "1" #케라스 2 쓸 거

In [66]:
import tensorflow_recommenders as tfrs

- 데이터 세트 준비

In [67]:
ratings = tfds.load("movielens/100k-ratings", split="train") # 일단 split="train"이라고 해서 몽땅 가져오기 (아직 분할 안 함)
movies = tfds.load("movielens/100k-movies", split="train")

In [68]:
for x in ratings.take(1).as_numpy_iterator(): # 넘파이 반복자로서 하나씩 꺼내준다.
    pprint.pprint(x)
    
# 딕셔너리 형태로 출력

{'bucketized_user_age': 45.0,
 'movie_genres': array([7]),
 'movie_id': b'357',
 'movie_title': b"One Flew Over the Cuckoo's Nest (1975)",
 'raw_user_age': 46.0,
 'timestamp': 879024327,
 'user_gender': True,
 'user_id': b'138',
 'user_occupation_label': 4,
 'user_occupation_text': b'doctor',
 'user_rating': 4.0,
 'user_zip_code': b'53211'}


2026-03-05 09:47:48.985770: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [69]:
for x in movies.take(1).as_numpy_iterator(): 
    pprint.pprint(x)

{'movie_genres': array([4]),
 'movie_id': b'1681',
 'movie_title': b'You So Crazy (1994)'}


2026-03-05 09:47:49.024621: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


- rating데이터에 초점 둘 것임

In [70]:
ratings = ratings.map(lambda x: {
    "movie_title" : x["movie_title"],
    "user_id" : x["user_id"],
})

movies = movies.map(lambda x: x["movie_title"])

In [71]:
for x in ratings.take(5).as_numpy_iterator():
    pprint.pprint(x)

{'movie_title': b"One Flew Over the Cuckoo's Nest (1975)", 'user_id': b'138'}
{'movie_title': b'Strictly Ballroom (1992)', 'user_id': b'92'}
{'movie_title': b'Very Brady Sequel, A (1996)', 'user_id': b'301'}
{'movie_title': b'Pulp Fiction (1994)', 'user_id': b'60'}
{'movie_title': b'Scream 2 (1997)', 'user_id': b'197'}


2026-03-05 09:47:49.094554: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


- 데이터 Train/ Test 분할

In [72]:
tf.random.set_seed(42) # 랜덤하게 섞는 규칙을 고정
# tfds의 묘미 "셔플", 데이터 10만개 섞기
shuffled = ratings.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

# 섞은 10만개 데이터중에 앞 8만개 훈련용/ 뒤에 2만개 테스트용
train = shuffled.take(80_000)
test = shuffled.skip(80_000).take(20_000)

In [73]:
movie_titles = movies.batch(1_000)
user_ids = ratings.batch(1_000_000).map(lambda x: x["user_id"])

# 위에서 천개 단위로 묶은 걸 리스트로 만들고 넘파이 배열로 길게 이어붙이고 중복 제거하고 가나다순 정렬까지해.
unique_movie_titles = np.unique(np.concatenate(list(movie_titles))) 
unique_user_ids = np.unique(np.concatenate(list(user_ids)))

unique_movie_titles[:10] # 앞에서부터 10개 보여줘, array: 넘파이 배열이라는 뜻

array([b"'Til There Was You (1997)", b'1-900 (1994)',
       b'101 Dalmatians (1996)', b'12 Angry Men (1957)', b'187 (1997)',
       b'2 Days in the Valley (1996)',
       b'20,000 Leagues Under the Sea (1954)',
       b'2001: A Space Odyssey (1968)',
       b'3 Ninjas: High Noon At Mega Mountain (1998)',
       b'39 Steps, The (1935)'], dtype=object)

- 모델 구현 (각 타워를 개별적으로 구축한 다음 최종 모델에서 결합)

1. 쿼리 타워

In [74]:
embedding_dimension = 32 
# 글자 데이터를 숫자의 뭉치로 바꾼다는 뜻.
# 유저 한 명을 설명할 때 32개의 숫자를 쓰겠다.

In [ ]:
user_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(
        vocabulary=unique_user_ids, mask_token=None),
    tf.keras.layers.Embedding(len(unique_user_ids)+1, embedding_dimension) # 유저수+1칸, 한 명당 32개자리 주삼.
])

2. 후보 타워

In [79]:
movie_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(
        vocabulary=unique_movie_titles, mask_token=None),
    tf.keras.layers.Embedding(len(unique_movie_titles)+1, embedding_dimension)
])

- 채점 설정

In [ ]:
metrics = tfrs.metrics.FactorizedTopK(
    candidates=movies.batch(128).map(movie_model)
)

- 손실 계산 설정

In [ ]:
task = tfrs.tasks.Retrieval(
    metrics=metrics
)
# Retrieval: 검색, task: 정답을 얼마나 못맞혔는지 벌점 매기는, 채점설정이랑 동일한 Topk를 연동.